### Sections
| | |
|---|---|
| **A** | Setup & load artefacts |
| **B** | Rebuild RAG pipeline |
| **C** | Strategy 1 — Zero-shot |
| **D** | Strategy 2 — Few-shot |
| **E** | Strategy 3 — Chain-of-Thought |
| **F** | Strategy 4 — Optimised System Prompt |
| **G** | Side-by-side comparison |


### Section A — Setup & Load Artifacts

In [ ]:
import os, sys, json, time, getpass, warnings
from pathlib import Path
from dotenv import load_dotenv
from typing  import List, Dict, Any, Optional

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sentence_transformers import SentenceTransformer
import torch, chromadb

from langchain_core.documents      import Document
from langchain_core.prompts        import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables      import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_core.messages       import HumanMessage, SystemMessage
from langchain_groq                import ChatGroq

warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

root = Path.cwd().parent
sys.path.insert(0, str(root))
from config import DATA_DIR

In [ ]:
# ── Path configuration ──────

OUTPUT_DIR = DATA_DIR / "outputs"
MODEL_DIR  = DATA_DIR / "models"

RAG_CONFIG_FILE = OUTPUT_DIR / "rag_config.json"
if not RAG_CONFIG_FILE.exists():
    raise FileNotFoundError("rag_config.json not found. Run Notebook 03_01 first.")

with open(RAG_CONFIG_FILE, "r") as f:
    cfg = json.load(f)

GROQ_MODEL_NAME  = cfg["groq_model_name"]
COLLECTION_NAME  = cfg["collection_name"]
FINE_TUNED_MODEL = cfg["embedding_model"]
N_RESULTS        = cfg["n_results_default"]
BASE_SYSTEM_PROMPT = cfg["system_prompt"]
FT_EMBEDDINGS    = OUTPUT_DIR / "ft_embeddings.npy"
CHUNK_METADATA   = OUTPUT_DIR / "chunk_metadata.json"

print(f"   Groq model : {GROQ_MODEL_NAME}")
print(f"   Chunks     : {cfg['total_chunks']}")


### Section B — Rebuild RAG Pipeline

In [ ]:
# Load embedding model
embedding_model = SentenceTransformer(FINE_TUNED_MODEL, device=DEVICE)

# Load pre-computed embeddings and metadata
ft_embeddings = np.load(FT_EMBEDDINGS)
with open(CHUNK_METADATA, "r") as f:
    metadata_list = json.load(f)
chunks       = [m["chunk"]       for m in metadata_list]
chunk_labels = [m["entity_type"] for m in metadata_list]

# Rebuild ChromaDB
try:
    chroma_client = chromadb.EphemeralClient()
except AttributeError:
    chroma_client = chromadb.Client()

existing = [c.name for c in chroma_client.list_collections()]
if COLLECTION_NAME in existing:
    chroma_client.delete_collection(COLLECTION_NAME)

collection = chroma_client.create_collection(
    name=COLLECTION_NAME, metadata={"hnsw:space": "cosine"}
)

BATCH = 200
for s in range(0, len(chunks), BATCH):
    e = min(s + BATCH, len(chunks))
    collection.add(
        ids        = [f"chunk_{i:04d}" for i in range(s, e)],
        documents  = chunks[s:e],
        embeddings = ft_embeddings[s:e].tolist(),
        metadatas  = [{"entity_type": chunk_labels[i], "chunk_index": i} for i in range(s, e)],
    )

print(f"ChromaDB rebuilt — {collection.count()} docs indexed.")


### B2 — Groq API Key & Core Functions

In [ ]:
# ── Load Groq API key from .env or prompt if needed ────────────────
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    GROQ_API_KEY = getpass.getpass("Enter Groq API key (hidden): ")
    print("Groq API key received via getpass.")
else:
    print("Groq API key loaded from .env")

if not GROQ_API_KEY.startswith("gsk_"):
    raise ValueError("Invalid key format. Expected 'gsk_...'")
print(f"   Key prefix : {GROQ_API_KEY[:8]}{'*'*20}")


In [ ]:
# ── Helper functions: retrieval and formatting ────────────────────────────────
def retrieve_documents(query: str, k: int = 5) -> List[Dict]:
    """
    Retrieve top-k documents from ChromaDB based on query.
    Returns list of dicts with 'chunk' and 'entity_type' keys.
    """
    results = collection.query(
        query_texts=[query],
        n_results=k,
    )
    
    docs = []
    if results and results['documents'] and len(results['documents']) > 0:
        for i, doc_text in enumerate(results['documents'][0]):
            metadata = results['metadatas'][0][i] if results['metadatas'] else {}
            docs.append({
                'chunk': doc_text,
                'entity_type': metadata.get('entity_type', 'Unknown'),
                'distance': results['distances'][0][i] if results['distances'] else None,
            })
    
    return docs

def format_documents(docs: List[Dict]) -> str:
    """
    Format retrieved documents as readable context for LLM.
    Formats as: "Context 1: [text]\nContext 2: [text]..."
    """
    if not docs:
        return "No relevant context found."
    
    formatted = []
    for i, doc in enumerate(docs, 1):
        chunk = doc.get('chunk', '')
        entity_type = doc.get('entity_type', 'Unknown')
        formatted.append(f"Context {i} ({entity_type}): {chunk}")
    
    return "\n".join(formatted)


### Section C — Strategy 1: Zero-Shot Prompting

In [ ]:
ZERO_SHOT_SYSTEM = """You are a supply chain analyst.
Answer the question using only the provided context.
If the context is insufficient, say so clearly."""

zero_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", ZERO_SHOT_SYSTEM),
    ("human", "{context}\n\nQuestion: {question}"),
])

llm_zero = ChatGroq(
    api_key=GROQ_API_KEY, model_name=GROQ_MODEL_NAME,
    temperature=0.1, max_tokens=1024
)

zero_shot_chain = (
    RunnableParallel(
        context  = RunnableLambda(lambda q: format_documents(retrieve_documents(q))),
        question = RunnablePassthrough()
    )
    | zero_shot_prompt | llm_zero | StrOutputParser()
)

BENCHMARK_QUERY = "Which suppliers have a lead time over 20 days and low reliability? What action should be taken?"

print("Running Zero-Shot...")
t0 = time.time()
zero_shot_answer = zero_shot_chain.invoke(BENCHMARK_QUERY)
zero_shot_time   = round(time.time() - t0, 2)

print(f"Done in {zero_shot_time}s")
print("\n--- ZERO-SHOT ANSWER ---")
print(zero_shot_answer)


### Section D — Strategy 2: Few-Shot Prompting

Provide 2 worked examples of (question → answer) pairs so the model learns the expected answer format and level of detail before seeing the actual question.

In [ ]:
FEW_SHOT_EXAMPLES = """
EXAMPLE 1
Question: Which warehouses are near capacity?
Answer:
Based on the knowledge graph:
• [Finding] Warehouse WH-03 (Chicago) is at 89% capacity — above the 85% critical threshold.
• [Finding] Warehouse WH-07 (Houston) is at 82% capacity — approaching warning level.
• [Action] Redistribute inventory from WH-03 to WH-05 (Phoenix, 54% load) immediately.
• [Source] Context 1, Context 3.

EXAMPLE 2
Question: Which routes have the highest transit delays?
Answer:
Based on the knowledge graph:
• [Finding] Route RT-004 (India → Los Angeles) has an average delay of 8 days.
• [Finding] Route RT-009 (China → New York) shows a 72% on-time rate — below target.
• [Action] Evaluate alternative carriers or routes for RT-004 and RT-009.
• [Source] Context 2, Context 4.
"""

FEW_SHOT_SYSTEM = f"""You are a supply chain analyst.
Answer questions using only the provided context.
Always follow this format:
• [Finding] specific observation with values from the context
• [Action]  recommended action with clear reasoning
• [Source]  cite which context entries support your answer

Here are two examples of correct answers:
{FEW_SHOT_EXAMPLES}"""

few_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", FEW_SHOT_SYSTEM),
    ("human", "{context}\n\nQuestion: {question}"),
])

llm_few = ChatGroq(
    api_key=GROQ_API_KEY, model_name=GROQ_MODEL_NAME,
    temperature=0.1, max_tokens=1024
)

few_shot_chain = (
    RunnableParallel(
        context  = RunnableLambda(lambda q: format_documents(retrieve_documents(q))),
        question = RunnablePassthrough()
    )
    | few_shot_prompt | llm_few | StrOutputParser()
)

print("Running Few-Shot...")
t0 = time.time()
few_shot_answer = few_shot_chain.invoke(BENCHMARK_QUERY)
few_shot_time   = round(time.time() - t0, 2)

print(f"Done in {few_shot_time}s")
print("\n--- FEW-SHOT ANSWER ---")
print(few_shot_answer)


### Section E — Strategy 3: Chain-of-Thought (CoT)

Instruct the model to reason step-by-step before giving its final answer. CoT improves accuracy on multi-step logistics decisions where intermediate reasoning matters.

In [ ]:
COT_SYSTEM = """You are a supply chain analyst.
When answering, always reason step-by-step before giving your final answer.

Use this structure:
Step 1 — Identify relevant entities from the context.
Step 2 — Analyse the data: compare values, find patterns, spot risks.
Step 3 — Draw conclusions from your analysis.
Step 4 — Recommend specific, actionable decisions.
Final Answer — Summarise in 2-3 sentences for the business team.

Use only information from the provided context."""

cot_prompt = ChatPromptTemplate.from_messages([
    ("system", COT_SYSTEM),
    ("human", "{context}\n\nQuestion: {question}\n\nReason step by step:"),
])

llm_cot = ChatGroq(
    api_key=GROQ_API_KEY, model_name=GROQ_MODEL_NAME,
    temperature=0.1, max_tokens=1500
)

cot_chain = (
    RunnableParallel(
        context  = RunnableLambda(lambda q: format_documents(retrieve_documents(q))),
        question = RunnablePassthrough()
    )
    | cot_prompt | llm_cot | StrOutputParser()
)

print("Running Chain-of-Thought...")
t0 = time.time()
cot_answer = cot_chain.invoke(BENCHMARK_QUERY)
cot_time   = round(time.time() - t0, 2)

print(f"Done in {cot_time}s")
print("\n--- CHAIN-OF-THOUGHT ANSWER ---")
print(cot_answer)


### Section F — Strategy 4: Optimised System Prompt

Combines the best elements of the previous strategies:
- **Role clarity** (from zero-shot)
- **Output format** (from few-shot)
- **Step-by-step reasoning** (from CoT)
- **Guardrails** for hallucination and source grounding

In [ ]:
OPTIMISED_SYSTEM = """You are a Senior Supply Chain AI Analyst supporting executive decisions.
Your knowledge comes exclusively from the Supply Chain Knowledge Graph provided as context.

REASONING PROCESS (internal — do not print these steps):
1. Scan context for entities relevant to the question.
2. Extract specific values (scores, quantities, dates, IDs).
3. Identify risks, bottlenecks, or opportunities.
4. Determine the best recommended action.

OUTPUT FORMAT (always use this structure):
**Summary:** One sentence direct answer.
**Key Findings:**
  • [Entity type | ID] finding with specific values
  • (repeat for each relevant finding)
**Recommended Action:** Specific, justified recommendation.
**Confidence:** High / Medium / Low — and why.

RULES:
- Use only information from the provided context.
- If context is insufficient, state confidence as Low and explain what is missing.
- Always reference context entries (e.g. Context 1) that support each finding."""

optimised_prompt = ChatPromptTemplate.from_messages([
    ("system", OPTIMISED_SYSTEM),
    ("human", "{context}\n\nQuestion: {question}"),
])

llm_opt = ChatGroq(
    api_key=GROQ_API_KEY, model_name=GROQ_MODEL_NAME,
    temperature=0.1, max_tokens=1024
)

optimised_chain = (
    RunnableParallel(
        context  = RunnableLambda(lambda q: format_documents(retrieve_documents(q))),
        question = RunnablePassthrough()
    )
    | optimised_prompt | llm_opt | StrOutputParser()
)

print("Running Optimised Prompt...")
t0 = time.time()
optimised_answer = optimised_chain.invoke(BENCHMARK_QUERY)
optimised_time   = round(time.time() - t0, 2)

print(f"Done in {optimised_time}s")
print("\n--- OPTIMISED ANSWER ---")
print(optimised_answer)


### Section G — Side-by-Side Comparison

We evaluate each strategy on three dimensions:
- **Structure** — does the answer follow a clear format?
- **Specificity** — does it cite values, IDs, entity names?
- **Actionability** — does it recommend a concrete next step?

In [ ]:
# Build comparison table
strategies = [
    {
        "name"        : "Zero-Shot",
        "answer"      : zero_shot_answer,
        "latency"     : zero_shot_time,
        "description" : "No examples, minimal instructions",
    },
    {
        "name"        : "Few-Shot",
        "answer"      : few_shot_answer,
        "latency"     : few_shot_time,
        "description" : "2 worked examples in prompt",
    },
    {
        "name"        : "Chain-of-Thought",
        "answer"      : cot_answer,
        "latency"     : cot_time,
        "description" : "Step-by-step reasoning required",
    },
    {
        "name"        : "Optimised Prompt",
        "answer"      : optimised_answer,
        "latency"     : optimised_time,
        "description" : "Role + format + CoT + guardrails",
    },
]

# Simple heuristic scores (1-5) based on answer characteristics
def score_answer(answer: str) -> Dict[str, int]:
    a = answer.lower()
    structure    = min(5, sum([
        2 if any(w in a for w in ["summary", "finding", "action", "step"]) else 0,
        2 if answer.count("•") + answer.count("-") > 2 else 0,
        1 if "**" in answer or "#" in answer else 0,
    ]))
    specificity  = min(5, sum([
        2 if any(c.isdigit() for c in answer) else 0,
        2 if any(w in a for w in ["context", "source", "sim="]) else 0,
        1 if len(answer) > 300 else 0,
    ]))
    actionability = min(5, sum([
        2 if any(w in a for w in ["recommend", "action", "should", "immediately"]) else 0,
        2 if any(w in a for w in ["redistribute", "evaluate", "contact", "prioritise", "review"]) else 0,
        1 if any(w in a for w in ["because", "reason", "due to", "since"]) else 0,
    ]))
    return {
        "Structure"    : structure,
        "Specificity"  : specificity,
        "Actionability": actionability,
        "Total"        : structure + specificity + actionability,
    }

rows = []
for s in strategies:
    scores = score_answer(s["answer"])
    rows.append({
        "Strategy"     : s["name"],
        "Description"  : s["description"],
        "Latency (s)"  : s["latency"],
        "Word Count"   : len(s["answer"].split()),
        **scores,
    })

df_comparison = pd.DataFrame(rows)
print("Strategy Comparison:")
print(df_comparison[["Strategy","Latency (s)","Word Count",
                      "Structure","Specificity","Actionability","Total"]].to_string(index=False))


In [ ]:
# Visualise comparison
fig = plt.figure(figsize=(16, 10))
fig.suptitle("Prompt Engineering Strategy Comparison\nSupply Chain RAG Pipeline",
             fontsize=14, fontweight="bold")

gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, :])

colors  = ["#95A5A6", "#4A90D9", "#9B59B6", "#7ED321"]
names   = df_comparison["Strategy"].tolist()
x       = np.arange(len(names))

# Radar-style: grouped bar for scores
metrics = ["Structure", "Specificity", "Actionability"]
w       = 0.25
for i, (metric, color) in enumerate(zip(metrics, ["#4A90D9","#7ED321","#E74C3C"])):
    ax1.bar(x + i*w - w, df_comparison[metric], w,
            label=metric, color=color, edgecolor="white")
ax1.set_xticks(x)
ax1.set_xticklabels(names, rotation=15, ha="right", fontsize=9)
ax1.set_ylim(0, 5)
ax1.set_ylabel("Score (0-5)")
ax1.set_title("Quality Scores by Dimension")
ax1.legend(fontsize=8)
ax1.grid(axis="y", alpha=0.3)

# Latency
ax2.bar(names, df_comparison["Latency (s)"], color=colors, edgecolor="white")
for i, v in enumerate(df_comparison["Latency (s)"]):
    ax2.text(i, v + 0.05, f"{v}s", ha="center", fontsize=9)
ax2.set_title("API Latency per Strategy")
ax2.set_ylabel("Seconds")
ax2.tick_params(axis="x", rotation=15)
ax2.grid(axis="y", alpha=0.3)

# Word count vs total score scatter
ax3.scatter(df_comparison["Word Count"], df_comparison["Total"],
            c=colors, s=200, zorder=5, edgecolors="white", linewidths=1.5)
for _, row in df_comparison.iterrows():
    ax3.annotate(row["Strategy"],
                 (row["Word Count"], row["Total"]),
                 textcoords="offset points", xytext=(8, 4), fontsize=9)
ax3.set_xlabel("Answer Word Count")
ax3.set_ylabel("Total Quality Score (max 15)")
ax3.set_title("Answer Length vs Quality — ideal = top-right")
ax3.grid(alpha=0.3)

plt.savefig(OUTPUT_DIR / "03_02_prompt_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Comparison chart saved → outputs/03_02_prompt_comparison.png")


In [ ]:
# Print all four answers side by side for the capstone report
print("=" * 70)
print("  FULL ANSWER COMPARISON — SAME QUERY, FOUR STRATEGIES")
print("=" * 70)
print(f"\nQuery: \"{BENCHMARK_QUERY}\"")

for s in strategies:
    print(f"\n{'━'*70}")
    print(f"  {s['name'].upper()}  |  {s['description']}  |  {s['latency']}s")
    print(f"{'━'*70}")
    print(s["answer"])


In [ ]:
# Save the optimised prompt for use in 4C
pe_output = {
    "benchmark_query"  : BENCHMARK_QUERY,
    "optimised_system" : OPTIMISED_SYSTEM,
    "best_strategy"    : "Optimised Prompt",
    "strategy_scores"  : rows,
}
PE_FILE = OUTPUT_DIR / "prompt_engineering_output.json"
with open(PE_FILE, "w") as f:
    json.dump(pe_output, f, indent=2)
